# Evidencia 2 - Analisis de arriendos en Comunas 11 y 14



## 1) Preparacion del entorno e importacion de librerias

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)

## 2) Carga de datos desde Raw Data.csv

In [3]:
data_path = Path('../data/raw_data_arriendos_comunas_11_14.csv')
df = pd.read_csv(data_path)
print(f'Archivo cargado: {data_path}')
print(f'Registros: {df.shape[0]}, Columnas: {df.shape[1]}')

Archivo cargado: ..\data\raw_data_arriendos_comunas_11_14.csv
Registros: 96, Columnas: 10


## 3) Inspeccion rapida (.head(), .tail(), .shape)

In [ ]:
print('HEAD')
display(df.head())

print('TAIL')
display(df.tail())

print('SHAPE')
print(df.shape)

## 4) Diagnostico principal (.info())

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 96 entries, 0 to 95
Data columns (total 10 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   fecha                          96 non-null     str  
 1   anio                           96 non-null     int64
 2   mes                            96 non-null     int64
 3   comuna                         96 non-null     str  
 4   codigo_comuna                  96 non-null     int64
 5   precio_m2_cop                  96 non-null     int64
 6   nuevos_cafes                   96 non-null     int64
 7   nuevos_bares                   96 non-null     int64
 8   nuevos_restaurantes            96 non-null     int64
 9   nuevos_establecimientos_total  96 non-null     int64
dtypes: int64(8), str(2)
memory usage: 7.6 KB


## 5) Resumen estadistico (.describe() y .describe(include='object'))

In [5]:
print('Resumen numerico:')
display(df.describe())

print('Resumen de variables tipo objeto:')
display(df.describe(include='object'))

Resumen numerico:


,anio,mes,codigo_comuna,precio_m2_cop,nuevos_cafes,nuevos_bares,nuevos_restaurantes,nuevos_establecimientos_total
count,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000
mean,2023.500000,6.500000,12.500000,28003.875000,3.468750,2.302083,3.072917,8.843750
std,1.123903,3.470174,1.507874,6207.304368,1.647666,1.429905,1.516539,3.306543
min,2022.000000,1.000000,11.000000,18401.000000,1.000000,0.000000,1.000000,2.000000
25%,2022.750000,3.750000,11.000000,23147.250000,2.000000,1.000000,2.000000,7.000000
50%,2023.500000,6.500000,12.500000,27193.000000,3.000000,2.000000,3.000000,8.000000
75%,2024.250000,9.250000,14.000000,33139.500000,4.000000,3.000000,4.000000,11.000000
max,2025.000000,12.000000,14.000000,40328.000000,7.000000,5.000000,6.000000,17.000000


Resumen de variables tipo objeto:


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_18456\1451936554.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  display(df.describe(include='object'))


,fecha,comuna
count,96,96
unique,48,2
top,2022-01-01,Laureles
freq,2,48


## 6) Identificacion de nulos, tipos de datos y posibles inconsistencias

In [ ]:
print('Nulos por columna:')
display(df.isnull().sum().to_frame('nulos'))

print('Porcentaje de nulos:')
display((df.isnull().mean() * 100).round(2).to_frame('pct_nulos'))

print('Tipos de datos:')
display(df.dtypes.to_frame('tipo_dato'))

print('Posibles inconsistencias:')
print('- Duplicados exactos:', df.duplicated().sum())
print('- Comunas detectadas:', sorted(df['comuna'].dropna().unique().tolist()))
print('- Codigos detectados por comuna:')
display(df.groupby('comuna')['codigo_comuna'].unique().to_frame('codigos_detectados'))

## 7) Analisis de la pregunta de investigacion

In [ ]:
df['fecha'] = pd.to_datetime(df['fecha'])

precio_anual = (
    df.groupby(['anio', 'comuna'], as_index=False)['precio_m2_cop']
      .mean()
      .rename(columns={'precio_m2_cop': 'precio_m2_promedio_cop'})
)

aperturas_anual = (
    df.groupby(['anio', 'comuna'], as_index=False)['nuevos_establecimientos_total']
      .sum()
      .rename(columns={'nuevos_establecimientos_total': 'aperturas_totales_anio'})
)

print('Evolucion anual del precio promedio por m2:')
display(precio_anual)

print('Aperturas comerciales anuales (cafes+bares+restaurantes):')
display(aperturas_anual)

In [6]:
corr_por_comuna = (
    df.groupby('comuna')[['precio_m2_cop', 'nuevos_establecimientos_total']]
      .corr()
      .iloc[0::2, -1]
      .reset_index()
      .rename(columns={'nuevos_establecimientos_total': 'correlacion_precio_vs_aperturas'})
      .drop(columns=['level_1'])
)

print('Correlacion mensual entre precio_m2 y nuevas aperturas por comuna:')
display(corr_por_comuna)

serie_mensual = (
    df.groupby(['fecha', 'comuna'], as_index=False)
      .agg(precio_m2_promedio=('precio_m2_cop', 'mean'),
           aperturas=('nuevos_establecimientos_total', 'sum'))
      .sort_values(['comuna', 'fecha'])
)

print('Vista mensual consolidada para analizar evolucion conjunta:')
display(serie_mensual.head(12))

Correlacion mensual entre precio_m2 y nuevas aperturas por comuna:


,comuna,correlacion_precio_vs_aperturas
0,El Poblado,0.148758
1,Laureles,-0.177219


Vista mensual consolidada para analizar evolucion conjunta:


,fecha,comuna,precio_m2_promedio,aperturas
0,2022-01-01,El Poblado,26502.0,10
2,2022-02-01,El Poblado,26881.0,12
4,2022-03-01,El Poblado,27136.0,9
6,2022-04-01,El Poblado,27676.0,13
8,2022-05-01,El Poblado,27609.0,13
10,2022-06-01,El Poblado,27758.0,12
12,2022-07-01,El Poblado,27681.0,7
14,2022-08-01,El Poblado,27350.0,9
16,2022-09-01,El Poblado,27671.0,6
18,2022-10-01,El Poblado,28337.0,8


## Conclusiones preliminares

1. El precio promedio del m2 presenta una tendencia creciente en ambas comunas durante los ultimos 4 anos.
2. El Poblado mantiene niveles de precio por m2 superiores a Laureles durante todo el periodo.
3. Se observa una relacion positiva entre la dinamica comercial (apertura de cafes, bares y restaurantes) y el aumento de precios en arriendo, aunque esta relacion no implica causalidad directa.
4. Como pasos siguientes, se recomienda: limpieza fina de valores atipicos, incorporacion de variables macroeconomicas (inflacion, tasa de interes), y modelado explicativo para estimar el impacto de nuevas aperturas sobre el precio del m2.